# MAVIS — Análisis de Probe Lineal Supervisado
## Motivación
### Datasets y encoders
### Anti-fuga
### Pipeline


## 1. Imports y configuración


In [ ]:
import sys, json, warnings
from pathlib import Path

# Add repo root to path
REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score

import experiments.analysis.mavis_analysis as ma

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

MODELS = ["ge2", "qw2b", "qw8b"]   # WAVE skipped (token bug / regen pending)
RESULTS = {}

print("REPO_ROOT:", REPO_ROOT)
print("FIG_DIR:", ma.FIG_DIR)


## 2. Helpers compartidos


In [ ]:
# ── Helpers del probe (protocolo unificado: 50×70/30 agrupado, AUC y accuracy out-of-fold) ──
N_REPS, TRAIN_FRAC, PROBE_C = 50, 0.7, 0.1   # sin PCA: dimensión completa con C bajo (el PCA rompe el Hadamard)


def l2_norm(v):
    n = np.linalg.norm(v)
    return v / n if n > 1e-9 else v


def _l2rows(M):
    n = np.linalg.norm(M, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return M / n


def hadamard(A, B):
    """Producto elemento a elemento de filas L2-normalizadas. Nota: sum(hadamard) = cos(A, B)."""
    return _l2rows(A) * _l2rows(B)


def _split_groups(groups, frac, rng):
    U = np.unique(groups)
    perm = U[rng.permutation(len(U))]
    train = set(perm[:int(round(frac * len(U)))].tolist())
    tr = np.fromiter((g in train for g in groups), bool, len(groups))
    return tr, ~tr


def _balance(idx, y, rng):
    pos = idx[y[idx] == 1]; neg = idx[y[idx] == 0]; m = min(len(pos), len(neg))
    if m == 0:
        return idx
    return np.concatenate([rng.choice(pos, m, replace=False), rng.choice(neg, m, replace=False)])


def eval_probe(X, y, groups, seed=42, balanced=False, n_reps=N_REPS, frac=TRAIN_FRAC, C=PROBE_C):
    """50×70/30 agrupado. Del MISMO clasificador entrenado se leen, fuera de fold, las dos métricas:
       accuracy = predict (umbral en decision_function=0, i.e. p=0.5); auc = roc sobre decision_function.
       balanced=True → subsample equilibrado por repetición (chance 0.5, comparable al umbral coseno)."""
    y = np.asarray(y, int); idx = np.arange(len(y)); accs, aucs = [], []
    for rep in range(n_reps):
        rng = np.random.default_rng(seed + rep)
        trm, tem = _split_groups(groups, frac, rng)
        tr, te = idx[trm], idx[tem]
        if balanced:
            tr = _balance(tr, y, rng); te = _balance(te, y, rng)
        if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
            continue
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("clf", LogisticRegression(max_iter=1000, C=C,
                                                    class_weight="balanced", solver="lbfgs"))])
        pipe.fit(X[tr], y[tr])
        s = pipe.decision_function(X[te])
        accs.append(float(((s > 0).astype(int) == y[te]).mean()))
        aucs.append(float(roc_auc_score(y[te], s)))
    base = 0.5 if balanced else max(float(np.mean(y)), 1.0 - float(np.mean(y)))
    return {"acc_mean": round(float(np.mean(accs)), 4), "acc_std": round(float(np.std(accs)), 4),
            "auc_mean": round(float(np.mean(aucs)), 4), "auc_std": round(float(np.std(aucs)), 4),
            "baseline": round(float(base), 4), "n": int(len(y))}


print("Helpers OK (eval_probe, hadamard)")


### Metodología del probe (unificada para los 3 datasets)


## 3. Experimento 1 — GroundLie360 Binary Probe


In [ ]:
RESULTS["GroundLie360"] = {}

# Anti-fuga: split por evento Snopes (título idéntico ∪ vídeo-fuente compartido) → dos vídeos
# del mismo fact-check no cruzan train/test e inflan el probe de título.
gl_events = ma.groundlie_event_ids()
print(f"  GL360 eventos (anti-fuga): {len(set(gl_events.values()))} grupos para {len(gl_events)} vídeos")

for model in MODELS:
    try:
        path = ma.db_path("GroundLie360", model)
        if not path.exists():
            print(f"  [{model}] DB not found, skipping."); continue
        conn = ma.connect(path)
        vid_emb   = ma.load_globals(conn, "video")
        title_emb = ma.load_globals(conn, "text_title")
        tr_emb    = ma.load_globals(conn, "transcript")
        labels_df = ma.load_labels(conn)
        conn.close()

        ids = sorted(set(vid_emb) & set(title_emb) & set(tr_emb) & set(labels_df.index))
        ids = [i for i in ids if pd.notna(labels_df.loc[i, "binary_label"])]
        y = np.array([int(labels_df.loc[i, "binary_label"]) for i in ids])
        groups = np.array([gl_events.get(i, i) for i in ids])
        V  = _l2rows(np.stack([vid_emb[i]   for i in ids]).astype(np.float64))
        T  = _l2rows(np.stack([title_emb[i] for i in ids]).astype(np.float64))
        Tr = _l2rows(np.stack([tr_emb[i]    for i in ids]).astype(np.float64))

        # baseline zero-shot de la arista (-cos(V,title)): alto = fake
        zs_auc = float(roc_auc_score(y, np.array([-ma.cos(vid_emb[i], title_emb[i]) for i in ids])))

        feats = {
            "video":               V,
            "title":               T,
            "transcript":          Tr,
            "concat":              np.hstack([V, T, Tr]),
            "relational_edge":     hadamard(V, T),
            "relational_triangle": np.hstack([hadamard(V, T), hadamard(V, Tr), hadamard(T, Tr)]),
        }
        RESULTS["GroundLie360"][model] = {}
        for fs, X in feats.items():
            res = eval_probe(X, y, groups, balanced=True)   # GL ~balanceado; subsample → chance 0.5
            res["zero_shot_auc"] = round(zs_auc, 4)
            RESULTS["GroundLie360"][model][fs] = res
            print(f"  GL360 [{model}] [{fs:20s}] AUC={res['auc_mean']:.3f}±{res['auc_std']:.3f}  "
                  f"ACC={res['acc_mean']:.3f}±{res['acc_std']:.3f}  base={res['baseline']:.3f}  N={res['n']}")
    except Exception as e:
        print(f"  [{model}] ERROR: {e}")
        import traceback; traceback.print_exc()

print("\nExperimento 1 completado.")


## 4. Experimento 2 — M3A OOC Probe (real vs MM-text)
### Anti-fuga


In [ ]:
m3a_list = json.loads((REPO_ROOT / "experiments/M3A/m3a_index_20000.json").read_text(encoding="utf-8"))
by_id = {item["raw_id"]: item for item in m3a_list}
print(f"M3A index loaded: {len(by_id)} entries")

RESULTS["M3A"] = {}
FS_M3A = ["video", "title", "transcript", "concat", "relational_edge", "relational_triangle"]
for model in MODELS:
    try:
        path = ma.db_path("M3A", model)
        if not path.exists():
            print(f"  [{model}] DB not found, skipping."); continue
        conn = ma.connect(path)
        vid_emb  = ma.load_globals(conn, "video")
        summ_emb = ma.load_globals(conn, "text_summary")
        tr_emb   = ma.load_globals(conn, "transcript")
        conn.close()

        # Pareado: por víctima rid → real (summary propio, 0) y fake MM (summary de otro evento, 1).
        # Vídeo y transcript son REALES en ambos (solo cambia el summary).
        V_, S_, Tr_, y_, g_ = [], [], [], [], []
        for rid, meta in by_id.items():
            src = (meta.get("mm_sources") or {}).get("text")
            if not src or rid not in vid_emb or rid not in summ_emb or rid not in tr_emb or src not in summ_emb:
                continue
            v, tr = vid_emb[rid], tr_emb[rid]
            V_ += [v, v]; Tr_ += [tr, tr]; S_ += [summ_emb[rid], summ_emb[src]]; y_ += [0, 1]; g_ += [rid, rid]
        if not y_:
            print(f"  [{model}] sin pares, skip."); continue
        V  = _l2rows(np.array(V_,  dtype=np.float32))   # float32 + construcción perezosa → memoria contenida
        S  = _l2rows(np.array(S_,  dtype=np.float32))
        Tr = _l2rows(np.array(Tr_, dtype=np.float32))
        y  = np.array(y_); groups = np.array(g_)
        del V_, S_, Tr_

        zs_auc = float(roc_auc_score(y, -np.einsum("ij,ij->i", V, S)))

        def _feat_m3a(name):                            # un solo feature set en memoria a la vez
            if name == "video":               return V
            if name == "title":               return S            # 'title' = summary (modalidad swapeada)
            if name == "transcript":          return Tr           # control degenerado (Tr idéntico) → ~0.5
            if name == "concat":              return np.hstack([V, S, Tr])
            if name == "relational_edge":     return hadamard(V, S)
            if name == "relational_triangle": return np.hstack([hadamard(V, S), hadamard(V, Tr), hadamard(S, Tr)])

        RESULTS["M3A"][model] = {}
        for fs in FS_M3A:
            X = _feat_m3a(fs)
            res = eval_probe(X, y, groups, balanced=False)   # pareado → ya balanceado
            del X
            res["zero_shot_auc"] = round(zs_auc, 4)
            RESULTS["M3A"][model][fs] = res
            print(f"  M3A [{model}] [{fs:20s}] AUC={res['auc_mean']:.3f}±{res['auc_std']:.3f}  "
                  f"ACC={res['acc_mean']:.3f}±{res['acc_std']:.3f}  N={res['n']}")
    except Exception as e:
        print(f"  [{model}] ERROR: {e}")
        import traceback; traceback.print_exc()

print("\nExperimento 2 completado.")


## 4b. Experimento 3 — FakeVV Probe (¿hay señal en la relación V·T más allá del coseno?)


In [ ]:
# Experimento 3 — FakeVV: ¿señal en la relación V·T más allá del coseno?
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

fakevv_index = json.loads((REPO_ROOT / "experiments/FakeVV_testset/google_embeddings2/test_index.json")
                          .read_text(encoding="utf-8"))

RESULTS["FakeVV"] = {}
for model in MODELS:
    try:
        path = ma.db_path("FakeVV", model)
        if not path.exists():
            print(f"  [{model}] DB not found, skipping."); continue
        conn = ma.connect(path)
        vid  = ma.load_globals(conn, "video")
        t_rl = ma.load_globals(conn, "text_real")
        t_fk = ma.load_globals(conn, "text_fake")
        conn.close()

        # Pareado: por vídeo r → título real (0) y título fake (1). El vídeo es el mismo en ambos.
        ids = sorted(set(vid) & set(t_rl) & set(t_fk))
        V_, T_, y_, g_ = [], [], [], []
        for r in ids:
            v = vid[r]
            V_ += [v, v]; T_ += [t_rl[r], t_fk[r]]; y_ += [0, 1]; g_ += [r, r]
        V = _l2rows(np.array(V_, dtype=np.float64))
        T = _l2rows(np.array(T_, dtype=np.float64))
        y = np.array(y_); groups = np.array(g_)

        zs_auc = float(roc_auc_score(y, np.array([-float(V[i] @ T[i]) for i in range(len(y))])))

        feats = {
            "video":           V,                       # control degenerado (V idéntico) → ~0.5
            "title":           T,                       # texto swapeado (real vs fake title)
            "concat":          np.hstack([V, T]),
            "relational_edge": hadamard(V, T),
        }
        RESULTS["FakeVV"][model] = {}
        for fs, X in feats.items():
            res = eval_probe(X, y, groups, balanced=False)   # pareado → ya balanceado
            res["zero_shot_auc"] = round(zs_auc, 4)
            RESULTS["FakeVV"][model][fs] = res
            print(f"  FakeVV [{model}] [{fs:16s}] AUC={res['auc_mean']:.3f}±{res['auc_std']:.3f}  "
                  f"ACC={res['acc_mean']:.3f}±{res['acc_std']:.3f}  N={res['n']}")
        print()
    except Exception as e:
        print(f"  [{model}] ERROR: {e}")
        import traceback; traceback.print_exc()

# Control 1: TF-IDF del TEXTO del título (sin embeddings) — referencia de sesgo léxico
texts, y_t, g_t = [], [], []
for e in fakevv_index:
    texts += [e["title"], e["fake_title"]]; y_t += [0, 1]; g_t += [e["raw_id"]] * 2
tfidf_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, sublinear_tf=True, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=0)),
])
proba_t = cross_val_predict(tfidf_pipe, texts, np.array(y_t),
                            cv=GroupKFold(5), groups=np.array(g_t), method="predict_proba")
print(f"  [control] TF-IDF texto del título: AUC={roc_auc_score(y_t, proba_t[:, 1]):.4f}")

# Control 2: ¿se reutilizan las entidades insertadas en los títulos fake?
swap_in = Counter()
for e in fakevv_index:
    tr = set(re.findall(r"[A-Za-z]+", e["title"].lower()))
    tf = set(re.findall(r"[A-Za-z]+", e["fake_title"].lower()))
    for w in tf - tr:
        swap_in[w] += 1
print("  [control] tokens insertados más repetidos:", swap_in.most_common(10))

print("\nExperimento 3 completado.")

## 4c. Experimento 3b — ¿Es la linealidad (o el clasificador) el límite?


In [ ]:
# Experimento 3b — ¿es la linealidad el límite? (FakeVV)
def probe_auc_3b(X, y, groups, clf="linear"):
    steps = [("scaler", StandardScaler())]
    if clf != "linear_full" and X.shape[1] > 512:
        steps.append(("pca", PCA(n_components=128, random_state=0)))
    if clf == "linear":
        steps.append(("clf", LogisticRegression(max_iter=2000, class_weight="balanced",
                                                random_state=0, solver="lbfgs")))
    elif clf == "linear_full":   # dimensión completa, más regularización
        steps.append(("clf", LogisticRegression(max_iter=3000, class_weight="balanced",
                                                random_state=0, C=0.1)))
    else:
        steps.append(("clf", HistGradientBoostingClassifier(random_state=0, max_iter=200)))
    proba = cross_val_predict(Pipeline(steps), X, y, cv=GroupKFold(5),
                              groups=groups, method="predict_proba")
    return float(roc_auc_score(y, proba[:, 1]))

rows_3b = []
for model in MODELS:
    info = ma.available("FakeVV", model)
    mods = info.get("modalities", {}) if info.get("exists") else {}
    if not (info.get("exists") and {"video", "text_real", "text_fake"} <= set(mods)):
        continue
    conn = ma.connect(ma.db_path("FakeVV", model))
    vid  = ma.load_globals(conn, "video")
    t_rl = ma.load_globals(conn, "text_real")
    t_fk = ma.load_globals(conn, "text_fake")
    conn.close()

    ids = sorted(set(vid) & set(t_rl) & set(t_fk))
    V_rows, T_rows, y_rows, g_rows = [], [], [], []
    for r in ids:
        v = l2_norm(vid[r].astype(np.float64))
        for t_emb, label in [(t_rl[r], 0), (t_fk[r], 1)]:
            V_rows.append(v)
            T_rows.append(l2_norm(t_emb.astype(np.float64)))
            y_rows.append(label)
            g_rows.append(r)
    V = np.stack(V_rows); T = np.stack(T_rows)
    y = np.array(y_rows, dtype=int); groups = np.array(g_rows)

    H = V * T                                   # hadamard: cos = H.sum(axis=1)
    cos_vt  = H.sum(axis=1)
    dist_vt = np.linalg.norm(V - T, axis=1)
    R2 = np.stack([cos_vt, dist_vt], axis=1)
    C  = np.concatenate([V, T], axis=1)
    zs = float(roc_auc_score(y, -cos_vt))

    res = {"model": model, "zs_cos": round(zs, 4)}
    for name, X, clf in [
        ("hadamard_lin", H, "linear"),        # bilineal diagonal aprendida (PCA128)
        ("hadamard_hgb", H, "hgb"),           # no-lineal sobre interacciones
        ("relational_hgb", R2, "hgb"),        # no-lineal sobre (cos, dist)
        ("concat_hgb", C, "hgb"),             # no-lineal sobre [V, T]
        ("title_hgb", T, "hgb"),              # control de sesgo, no-lineal
        ("hadamard_full", H, "linear_full"),  # bilineal a dimensión COMPLETA (sin PCA)
        ("title_full", T, "linear_full"),     # su control
    ]:
        auc = probe_auc_3b(X, y, groups, clf)
        res[name] = round(auc, 4)
        print(f"  [{model}] {name:<15} AUC={auc:.4f}")
    rows_3b.append(res)
    print()

df_3b = pd.DataFrame(rows_3b)
print(df_3b.to_string(index=False))

## 5. Guardar resultados


In [ ]:
OUT_JSON = REPO_ROOT / "experiments/analysis/probe_results.json"
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False)
print("Saved:", OUT_JSON)


## 6. Tabla resumen — AUC Probe vs Zero-shot


In [ ]:
rows = []
for dataset, enc_dict in RESULTS.items():
    for encoder, fs_dict in enc_dict.items():
        best_fs = max(fs_dict, key=lambda k: fs_dict[k]["auc_mean"])
        best = fs_dict[best_fs]
        rows.append({
            "Dataset":   dataset,
            "Encoder":   ma.MODEL_DISPLAY.get(encoder, encoder),
            "Best FS":   best_fs,
            "Probe AUC": best["auc_mean"],
            "ZS AUC":    best["zero_shot_auc"],
            "Delta":     round(best["auc_mean"] - best["zero_shot_auc"], 4),
            "Acc":       best["acc_mean"],
            "Baseline":  best["baseline"],
            "N":         best["n"],
        })

df_summary = pd.DataFrame(rows)
df_summary = df_summary.sort_values(["Dataset", "Probe AUC"], ascending=[True, False])
print(df_summary.to_string(index=False))


## 7. Figura — Probe AUC vs Zero-shot AUC


In [ ]:
with plt.rc_context(ma._RC):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

    for ax, dataset in zip(axes, ["GroundLie360", "M3A"]):
        enc_dict = RESULTS.get(dataset, {})
        encoders = [m for m in MODELS if m in enc_dict]
        if not encoders:
            ax.set_title(dataset + " (no data)")
            continue

        x = np.arange(len(encoders))
        w = 0.35
        probe_vals = []
        zs_vals = []
        for enc in encoders:
            fs_dict = enc_dict[enc]
            best_fs = max(fs_dict, key=lambda k: fs_dict[k]["auc_mean"])
            probe_vals.append(fs_dict[best_fs]["auc_mean"])
            zs_vals.append(fs_dict[best_fs]["zero_shot_auc"])

        bars1 = ax.bar(x - w/2, probe_vals, w, label="Probe (best FS)",
                       color=[ma.MODEL_COLOR.get(m, "#888") for m in encoders],
                       edgecolor="white", linewidth=0.6)
        bars2 = ax.bar(x + w/2, zs_vals, w, label="Zero-shot (cosine)",
                       color=[ma.MODEL_COLOR.get(m, "#888") for m in encoders],
                       edgecolor="black", linewidth=0.8, alpha=0.55, hatch="//")

        for bar, v in zip(bars1, probe_vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=7)
        for bar, v in zip(bars2, zs_vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=7)

        ax.set_xticks(x)
        ax.set_xticklabels([ma.MODEL_DISPLAY.get(m, m) for m in encoders], rotation=15, ha="right")
        ax.set_ylabel("ROC-AUC")
        ax.set_title(f"{dataset} — Probe vs Zero-shot")
        ax.axhline(0.5, color="#E74C3C", lw=1.2, ls="--", alpha=0.7, label="Random (0.5)")
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1.05)

    plt.tight_layout()
    fig_path = ma.FIG_DIR / "probe_vs_zeroshot.png"
    fig.savefig(str(fig_path), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)


## 8. Figura — Comparación de Feature Sets (qw8b, GroundLie360)


In [ ]:
ref_model = "qw8b"
gl_data = RESULTS.get("GroundLie360", {}).get(ref_model, {})

if gl_data:
    fs_names = list(gl_data.keys())
    auc_vals = [gl_data[fs]["auc_mean"] for fs in fs_names]
    zs_auc_ref = gl_data[fs_names[0]]["zero_shot_auc"]

    with plt.rc_context(ma._RC):
        fig, ax = plt.subplots(figsize=(8, 4))
        color = ma.MODEL_COLOR.get(ref_model, "#ED7D31")
        bars = ax.bar(fs_names, auc_vals, color=color, edgecolor="white", linewidth=0.6, alpha=0.85)
        for bar, v in zip(bars, auc_vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=8)
        ax.axhline(zs_auc_ref, color="#E74C3C", lw=1.4, ls="--", alpha=0.85,
                   label=f"Zero-shot cosine ({zs_auc_ref:.3f})")
        ax.axhline(0.5, color="gray", lw=0.8, ls=":", alpha=0.6, label="Random (0.5)")
        ax.set_ylabel("ROC-AUC")
        ax.set_title(f"GroundLie360 — Feature set comparison ({ma.MODEL_DISPLAY.get(ref_model, ref_model)})")
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1.05)
        plt.tight_layout()
        fig_path2 = ma.FIG_DIR / "probe_featuresets.png"
        fig.savefig(str(fig_path2), dpi=150, bbox_inches="tight")
        plt.show()
        print("Saved:", fig_path2)
else:
    print(f"No data for {ref_model} in GroundLie360, skipping figure.")


## 9. Interpretación
### ¿El probe supera al zero-shot?
### Auditoría de sesgo (2026-06) — qué mide realmente el probe en GroundLie360
### FakeVV — ¿relación V·T más allá del coseno? (Experimento 3)
### Dataset con señal presente pero zero-shot bajo
### Caveats
